In [1]:
import os
import re
import cv2
import math
import numpy as np
import pytesseract
import tensorflow as tf
from pdf2image import convert_from_path
from PIL import Image
from collections import defaultdict
from openpyxl import Workbook
from openpyxl.styles import Alignment
from openpyxl.drawing.image import Image as XLImage
from tensorflow.keras.models import load_model
from keras.saving import register_keras_serializable

pytesseract.pytesseract.tesseract_cmd = "/opt/homebrew/bin/tesseract"

PDF_PATH = "/Users/mahmudurrahman/Desktop/Project/Templates/b-scan.pdf"
OUTPUT_EXCEL = "/Users/mahmudurrahman/Desktop/Project/Templates/b.xlsx"
MODEL_PATH = "/Users/mahmudurrahman/Desktop/Project/Models/OnlyGerman.keras"

IMAGE_DIR = "excel_images"
SEGMENTATION_DIR = "segmentation_images"

DPI = 300
HEADER_HEIGHT_RATIO = 0.15
MAX_CHARS = 7

# questions that are decimal fields (0-indexed)
DECIMAL_QUESTION_INDICES = {2, 5, 8}

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(SEGMENTATION_DIR, exist_ok=True)

Image.MAX_IMAGE_PIXELS = 400_000_000
#CUSTOM SLICER FUNCTION (for model loading)
@register_keras_serializable(package="custom")
def slicer(x, idx=0, slice_size=28, axis=None):
    """Custom slicer function for Lambda layers in the model."""
    x = tf.convert_to_tensor(x)
    rank = len(x.shape)

    if rank == 4:
        if axis is None:
            s = x.shape
            if s[1] == 3 and s[2] == 196:      # channels_first
                axis = 2
            elif s[3] == 3 and s[1] == 196:    # channels_last
                axis = 1
            else:
                axis = 2

        if axis == 1:
            return x[:, idx, :, :]
        elif axis == 2:
            start = idx * slice_size
            return x[:, :, start:start + slice_size, :]
        else:
            return tf.gather(x, idx, axis=axis)

    return x

model = load_model(MODEL_PATH, custom_objects={"slicer": slicer})
idx_to_char = {**{i: str(i) for i in range(10)}, 10: ",", 11: ".", 12: ""}
# PDF PROCESSING
def pdf_to_images(pdf_path, dpi=300):
    pages = convert_from_path(pdf_path, dpi=dpi)
    images = []
    for p in pages:
        img = np.array(p)
        bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        images.append(bgr)
    return images

def preprocess_printed_header_for_ocr(header_bgr):
    gray = cv2.cvtColor(header_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.fastNlMeansDenoising(gray, h=15)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    bin_img = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 10
    )
    return bin_img


def parse_header_line(text):
    lastname = firstname = printed_id = ""
    page_no = None
    total_pages = None

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    joined = " | ".join(lines)
    joined = re.sub(r"\s+", " ", joined)

    pattern = re.compile(
        r"(?P<last>[^,|]+)\s*,\s*(?P<first>[^,|]+)\s*,\s*(?P<id>\d{4,})\s*,\s*Seite\s*(?P<p>\d+)(?:\s*/\s*(?P<t>\d+))?",
        re.IGNORECASE
    )

    m = pattern.search(joined)
    if m:
        lastname = m.group("last").strip()
        firstname = m.group("first").strip()
        printed_id = m.group("id").strip()
        page_no = int(m.group("p"))
        if m.group("t"):
            total_pages = int(m.group("t"))

    return lastname, firstname, printed_id, page_no, total_pages


def extract_printed_header_info(page_bgr, page_index):
    h, w = page_bgr.shape[:2]
    header = page_bgr[:int(h * HEADER_HEIGHT_RATIO), :]

    pre = preprocess_printed_header_for_ocr(header)

    cfgs = [
        "--oem 1 --psm 6",
        "--oem 1 --psm 7",
    ]

    best = ("", "", "", None, None)
    best_score = -1

    for cfg in cfgs:
        txt = pytesseract.image_to_string(pre, config=cfg)
        lastname, firstname, printed_id, page_no, total_pages = parse_header_line(txt)

        score = 0
        if printed_id: score += 50
        if lastname and firstname: score += 20
        if page_no is not None: score += 10
        score += min(len(txt), 200) * 0.01

        if score > best_score:
            best_score = score
            best = (lastname, firstname, printed_id, page_no, total_pages)

    lastname, firstname, printed_id, page_no, total_pages = best

    return {
        "lastname": lastname,
        "firstname": firstname,
        "printed_id": printed_id,
        "page_no": page_no,
        "total_pages": total_pages,
        "page_index": page_index,
    }
def detect_answer_boxes(image, require_handwriting=False):
    H, W = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    bin_img = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        41, 10
    )

    bin_img = cv2.morphologyEx(
        bin_img,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)),
        iterations=1
    )

    cnts, _ = cv2.findContours(bin_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)

        if w < W * 0.20:
            continue
        if h < H * 0.015:
            continue
        if h > H * 0.40:
            continue

        aspect = w / float(h + 1e-6)
        if not (aspect > 1.3 or h > H * 0.08):
            continue

        boxes.append((x, y, w, h))

    mid_x = W / 2.0
    left_col = [b for b in boxes if (b[0] + b[2] / 2) < mid_x]
    right_col = [b for b in boxes if (b[0] + b[2] / 2) >= mid_x]

    left_col = sorted(left_col, key=lambda b: b[1])
    right_col = sorted(right_col, key=lambda b: b[1])

    print(f"[INFO] Detected {len(boxes)} boxes")
    return left_col + right_col

def has_clear_vertical_gap(binary, c1, c2):
    """
    Returns True if there is a clear vertical whitespace gap between two blobs.
    """
    x_start = c1["x"] + c1["w"]
    x_end = c2["x"]

    if x_end <= x_start:
        return False

    y1 = max(c1["y"], c2["y"])
    y2 = min(c1["y"] + c1["h"], c2["y"] + c2["h"])

    if y2 <= y1:
        return True

    gap = binary[y1:y2, x_start:x_end]
    ink_ratio = np.sum(gap > 0) / gap.size

    return ink_ratio < 0.05

def has_clear_horizontal_gap(binary, c1, c2):
    """
    True only if there is a REAL horizontal separator
    (gap spans most of the digit height).
    """
    x1 = c1["x"] + c1["w"]
    x2 = c2["x"]

    if x2 <= x1:
        return False

    # vertical overlap region
    y_top = max(c1["y"], c2["y"])
    y_bot = min(c1["y"] + c1["h"], c2["y"] + c2["h"])

    gap_h = y_bot - y_top
    if gap_h <= 0:
        return False

    gap = binary[y_top:y_bot, x1:x2]
    if gap.size == 0:
        return False

    ink_ratio = np.sum(gap > 0) / gap.size

    # 🔥 KEY FIX:
    # gap must cover MOST of character height
    height_ratio = gap_h / min(c1["h"], c2["h"])

    return ink_ratio < 0.03 and height_ratio > 0.75



def merge_fragments(comps, binary, roi_h):
    if len(comps) <= 1:
        return comps

    comps = sorted(comps, key=lambda c: c["x"])
    merged = []
    cur = comps[0].copy()

    avg_h = np.mean([c["h"] for c in comps])

    for nxt in comps[1:]:

        # never merge punctuation
        if cur.get("is_punct", False) or nxt.get("is_punct", False):
            merged.append(cur)
            cur = nxt.copy()
            continue

        gap = nxt["x"] - (cur["x"] + cur["w"])

        y_overlap = min(
            cur["y"] + cur["h"],
            nxt["y"] + nxt["h"]
        ) - max(cur["y"], nxt["y"])

        should_merge = (
            gap < max(2, avg_h * 0.08) and
            y_overlap > avg_h * 0.55 and
            has_clear_vertical_gap(binary, cur, nxt) and
            not has_clear_horizontal_gap(binary, cur, nxt)
        )


        # never merge tall blobs
        if cur["h"] > 0.6 * roi_h or nxt["h"] > 0.6 * roi_h:
            should_merge = False

        if should_merge:
            x1 = min(cur["x"], nxt["x"])
            y1 = min(cur["y"], nxt["y"])
            x2 = max(cur["x"] + cur["w"], nxt["x"] + nxt["w"])
            y2 = max(cur["y"] + cur["h"], nxt["y"] + nxt["h"])

            cur = {
                "x": x1,
                "y": y1,
                "w": x2 - x1,
                "h": y2 - y1,
                "area": cur["area"] + nxt["area"],
                "is_punct": False
            }
        else:
            merged.append(cur)
            cur = nxt.copy()

    merged.append(cur)
    return merged




def split_wide_boxes(binary, comps):
    out = []

    for c in comps:
        x, y, w, h = c["x"], c["y"], c["w"], c["h"]

        # never split punctuation or small digits
        if c.get("is_punct", False) or h < 18 or w < 2.5 * h:
            out.append(c)
            continue

        roi = binary[y:y+h, x:x+w]
        proj = np.sum(roi > 0, axis=0) / h

        valleys = np.where(proj < 0.05)[0]
        if len(valleys) < 6:
            out.append(c)
            continue

        splits = np.split(valleys, np.where(np.diff(valleys) != 1)[0] + 1)
        best = max(splits, key=len)
        split_x = int(best[len(best) // 2])

        left = {
            "x": x,
            "y": y,
            "w": split_x,
            "h": h,
            "area": split_x * h
        }
        right = {
            "x": x + split_x,
            "y": y,
            "w": w - split_x,
            "h": h,
            "area": (w - split_x) * h
        }

        out.extend([left, right])

    return out



def merge_decimal_into_neighbors(comps):
    if len(comps) <= 1:
        return comps

    out = []
    i = 0

    while i < len(comps):
        c = comps[i]

        if not c.get("is_decimal", False):
            out.append(c)
            i += 1
            continue

        cx = c["x"] + c["w"] / 2

        left = out[-1] if out else None
        right = comps[i + 1] if i + 1 < len(comps) else None

        candidates = [
            d for d in (left, right)
            if d and not d.get("is_punct", False)
        ]

        if not candidates:
            out.append(c)
            i += 1
            continue

        nearest = min(
            candidates,
            key=lambda d: abs((d["x"] + d["w"] / 2) - cx)
        )

        x1 = min(nearest["x"], c["x"])
        y1 = min(nearest["y"], c["y"])
        x2 = max(nearest["x"] + nearest["w"], c["x"] + c["w"])
        y2 = max(nearest["y"] + nearest["h"], c["y"] + c["h"])

        nearest.update({
            "x": x1,
            "y": y1,
            "w": x2 - x1,
            "h": y2 - y1,
            "area": nearest["area"] + c["area"]
        })

        i += 1  # skip dot

    return out

#  PUNCTUATION SIZE WINDOW 
PUNCT_W_MIN, PUNCT_W_MAX = 8, 35
PUNCT_H_MIN, PUNCT_H_MAX = 14, 69
PUNCT_A_MIN, PUNCT_A_MAX = 108, 647

def mark_punctuation_by_stats(comps):

    if len(comps) < 2:
        for c in comps:
            c["is_punct"] = False
        return comps

    #  group reference 
    median_area = float(np.median([c["area"] for c in comps]))
    median_h    = float(np.median([c["h"] for c in comps]))
    median_w    = float(np.median([c["w"] for c in comps]))

    #  estimate baseline from bottoms of "normal" (non-tiny) blobs 
    big = [c for c in comps if c["h"] >= 0.75 * median_h and c["area"] >= 0.60 * median_area]
    if not big:
        big = comps
    baseline = float(np.median([c["y"] + c["h"] for c in big]))
    top_line = float(np.percentile([c["y"] for c in big], 20))

    for c in comps:
        w, h, a, y = c["w"], c["h"], c["area"], c["y"]
        bottom = y + h

        
        # A) STATS WINDOW
         
        in_window = (
            PUNCT_W_MIN <= w <= PUNCT_W_MAX and
            PUNCT_H_MIN <= h <= PUNCT_H_MAX and
            PUNCT_A_MIN <= a <= PUNCT_A_MAX
        )

     
        # B) TINY DOT (micro decimal) (handles 2x3, 3x4 etc)
        
        tiny_dot = (
            a < 0.12 * median_area and
            h < 0.35 * median_h and
            w < 0.35 * median_w
        )

        # POSITION RULE 
        # punctuation sits low/near baseline
        
        near_baseline = bottom > (baseline - 0.25 * median_h)
        starts_lower  = y > (top_line + 0.10 * median_h)

        # safety: never mark tall blobs as punctuation
        not_digit_like = h < 0.85 * median_h

        c["is_punct"] = (in_window or tiny_dot) and near_baseline and starts_lower and not_digit_like

    return comps

def enforce_single_punctuation(comps):
    """
    Enforce rule: two punctuations cannot be consecutive.
    If multiple punctuation blobs appear in sequence,
    keep ONLY the most plausible one.
    """
    if len(comps) < 2:
        return comps

    out = []
    i = 0

    while i < len(comps):
        c = comps[i]

        if not c.get("is_punct", False):
            out.append(c)
            i += 1
            continue

        # collect consecutive punctuations
        puncts = [c]
        j = i + 1
        while j < len(comps) and comps[j].get("is_punct", False):
            puncts.append(comps[j])
            j += 1

        if len(puncts) == 1:
            out.append(puncts[0])
        else:
            # 🔥 choose best punctuation
            # priority:
            # 1) larger area (real dot/comma beats pen tail)
            # 2) closer to baseline
            best = max(
                puncts,
                key=lambda p: (
                    p["area"],
                    p["y"] + p["h"]
                )
            )
            out.append(best)

        i = j

    return out

def normalize_char_image(binary, c, size=28, padding=4):
    x, y, w, h = c["x"], c["y"], c["w"], c["h"]
    crop = binary[y:y+h, x:x+w]

    if crop.size == 0:
        return np.ones((size, size), dtype=np.uint8) * 255

    # ink black on white
    crop = 255 - crop
    canvas = np.ones((size, size), dtype=np.uint8) * 255

    # -- PUNCTUATION PATH --
    if c.get("is_punct", False):
        hh, ww = crop.shape

        # target max dimension = size//4 (≈7px for 28)
        max_dim = size // 4

        # scale down if needed
        if hh > max_dim or ww > max_dim:
            scale = min(max_dim / float(hh), max_dim / float(ww))
            new_h = max(1, int(hh * scale))
            new_w = max(1, int(ww * scale))
            crop = cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_AREA)
        else:
            new_h, new_w = hh, ww

        # center horizontally, bottom align vertically (near bottom)
        x_offset = (size - new_w) // 2
        y_offset = size - new_h - padding

        x_offset = max(0, min(x_offset, size - new_w))
        y_offset = max(0, min(y_offset, size - new_h))

        canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = crop
        return canvas

    # -- DIGIT / NORMAL PATH --
    hh, ww = crop.shape
    target_max_dim = size - 2 * padding

    scale = target_max_dim / max(hh, ww + 1e-6)
    new_w = max(1, int(ww * scale))
    new_h = max(1, int(hh * scale))

    interpolation = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
    crop = cv2.resize(crop, (new_w, new_h), interpolation=interpolation)

    y0 = (size - new_h) // 2
    x0 = (size - new_w) // 2

    y0 = max(0, min(y0, size - new_h))
    x0 = max(0, min(x0, size - new_w))

    canvas[y0:y0 + new_h, x0:x0 + new_w] = crop
    return canvas


def remove_tall_thin_strokes(comps):
    """
    Removes vertical-stroke junk that is tall like digits but too thin.
    This specifically fixes cases like '2115' where a thin line appears far away.
    """
    if len(comps) < 3:
        return comps

    hs = np.array([c["h"] for c in comps], dtype=float)
    ws = np.array([c["w"] for c in comps], dtype=float)
    areas = np.array([c["area"] for c in comps], dtype=float)

    med_h = float(np.median(hs))
    med_w = float(np.median(ws))
    med_a = float(np.median(areas))

    filtered = []
    for c in comps:
        h, w, a = c["h"], c["w"], c["area"]

        # tall like digits, but very thin and low ink area => junk stroke
        tall = h >= 0.85 * med_h
        very_thin = w <= 0.35 * med_w
        low_area = a <= 0.45 * med_a

        if tall and very_thin and low_area and (not c.get("is_punct", False)):
            continue

        filtered.append(c)

    return filtered

def keep_main_cluster(comps):
    """
    Keep main digit cluster:
    - Vertical: keep components whose center_y is within ±0.4*median_digit_height of the main digit band
    - Horizontal: keep components inside the main digit span (xmin..xmax) with a small margin
    - Always allow punctuation if it is near baseline and inside the x-span
    """
    if len(comps) < 3:
        return comps

    # Use "digit-like" comps to define the main band (ignore tiny dots/noise)
    areas = np.array([c["area"] for c in comps], dtype=float)
    hs    = np.array([c["h"] for c in comps], dtype=float)
    ws    = np.array([c["w"] for c in comps], dtype=float)

    med_a = float(np.median(areas))
    med_h = float(np.median(hs))
    med_w = float(np.median(ws))

    digit_like = [
        c for c in comps
        if (not c.get("is_punct", False)) and c["h"] >= 0.70 * med_h and c["area"] >= 0.50 * med_a
    ]
    if len(digit_like) < 2:
        # not enough evidence to safely filter
        return comps

    #  main vertical band (center-based) 
    digit_cy = np.array([c["y"] + c["h"] / 2 for c in digit_like], dtype=float)
    main_cy  = float(np.median(digit_cy))
    band     = 0.40 * med_h   # <--  requirement

    # baseline for punctuation allowance
    baseline = float(np.median([c["y"] + c["h"] for c in digit_like]))

    #  main horizontal span from real digits 
    xmin = min(c["x"] for c in digit_like)
    xmax = max(c["x"] + c["w"] for c in digit_like)
    margin = 1.20 * med_w   # small safe margin so we DON'T drop first/last digit

    kept = []
    for c in comps:
        cx = c["x"] + c["w"] / 2
        cy = c["y"] + c["h"] / 2
        bottom = c["y"] + c["h"]

        in_x = (xmin - margin) <= cx <= (xmax + margin)

        if c.get("is_punct", False):
            # punctuation: allow if near baseline + inside x-span
            near_baseline = bottom >= (baseline - 0.35 * med_h)
            if in_x and near_baseline:
                kept.append(c)
            continue

        # normal digits: must be in x-span and inside vertical band
        in_y = abs(cy - main_cy) <= band
        if in_x and in_y:
            kept.append(c)

    # safety: if filtering becomes too aggressive, fallback
    if len(kept) >= max(2, len(digit_like)):
        return kept
    return comps

def robust_character_segmentation(roi_bgr, is_decimal_field=False):

    if roi_bgr is None or roi_bgr.size == 0:
        return [], None, []

    H, W = roi_bgr.shape[:2]

    # helper: adaptive binarize (ink = 255, background = 0)
    def _binarize(roi_bgr):
        gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
        gray = cv2.fastNlMeansDenoising(gray, h=15)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)

        block = max(31, (H // 10) | 1)  # odd
        bin_img = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY_INV,
            block, 9
        )
        return bin_img

    # helper: remove box border lines ONLY near ROI edges

    def _remove_border_lines(binary):
        """
        Detect long horizontal/vertical lines and erase them,
        but only inside a small edge band to avoid deleting digit strokes.
        """
        if binary is None or binary.size == 0:
            return binary

        edge = max(3, int(0.12 * H))   # edge band (top/bottom/left/right)

        # long line detectors
        hK = max(15, W // 10)
        vK = max(15, H // 10)
        horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (hK, 1))
        vert_kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (1, vK))

        # detect lines
        horiz = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horiz_kernel, iterations=1)
        vert  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vert_kernel,  iterations=1)

        line_mask = cv2.bitwise_or(horiz, vert)

        # thicken line mask a bit so merged punctuation detaches cleanly
        line_mask = cv2.dilate(line_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), iterations=1)

        # keep only near edges (so we don't remove digit strokes in the middle)
        edge_mask = np.zeros_like(binary)
        edge_mask[:edge, :] = 255
        edge_mask[H-edge:, :] = 255
        edge_mask[:, :edge] = 255
        edge_mask[:, W-edge:] = 255

        line_mask_edge = cv2.bitwise_and(line_mask, edge_mask)

        # erase detected edge-lines from binary
        cleaned = binary.copy()
        cleaned[line_mask_edge > 0] = 0
        return cleaned

    
    # helper: connected components -> list[dict]
    def _components_from_binary(binary):
        n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
        comps = []
        for i in range(1, n):
            x, y, w, h, area = stats[i]
            comps.append({"x": int(x), "y": int(y), "w": int(w), "h": int(h), "area": int(area)})
        return comps

    # helper: score whether a component is "line-like" (border junk)
    def _is_line_like(c):
        w, h, a = c["w"], c["h"], c["area"]
        aspect = w / (h + 1e-6)
        fill = a / (w * h + 1e-6)

        long_h = (w > 0.55 * W and h < 0.18 * H)
        long_v = (h > 0.55 * H and w < 0.18 * W)

        return (long_h or long_v) and fill < 0.35

    # helper: border touch (looser for punctuation cases)
    def _touches_border(c, margin=2):
        return (
            c["x"] <= margin or
            c["y"] <= margin or
            (c["x"] + c["w"]) >= (W - margin) or
            (c["y"] + c["h"]) >= (H - margin)
        )

    # helper: small blob removal using AREA + H + W (2-of-3 rule)
    def _remove_small_blobs_area_hw(comps):
        if len(comps) < 3:
            return comps

        areas   = np.array([c["area"] for c in comps], dtype=float)
        heights = np.array([c["h"] for c in comps], dtype=float)
        widths  = np.array([c["w"] for c in comps], dtype=float)

        med_a = float(np.median(areas))
        med_h = float(np.median(heights))
        med_w = float(np.median(widths))

        out = []
        for c in comps:
            if c.get("is_punct", False):
                out.append(c)
                continue

            a, h, w = c["area"], c["h"], c["w"]

            too_small_area = a < 0.35 * med_a
            too_short      = h < 0.50 * med_h
            too_narrow     = w < 0.50 * med_w

            small_score = int(too_small_area) + int(too_short) + int(too_narrow)

            # remove only if small in >=2 ways
            if small_score >= 2:
                continue

            out.append(c)
        return out

    # 1) BINARIZE + REMOVE BORDER LINES
    binary = _binarize(roi_bgr)
    binary = _remove_border_lines(binary)

    # 2) CONNECTED COMPONENTS
    comps = _components_from_binary(binary)
    if not comps:
        return [], binary, []

    # 3) BASIC FILTERS (DON'T kill border-touching yet)
    roi_area = H * W
    min_area = max(4, int(roi_area * 0.00005))
    max_area = int(roi_area * 0.75)

    valid = []
    for c in comps:
        if c["area"] < min_area or c["area"] > max_area:
            continue

        aspect = c["w"] / (c["h"] + 1e-6)
        if aspect < 0.08 or aspect > 6.0:
            continue

        c["touches_border"] = _touches_border(c, margin=2)
        valid.append(c)

    comps = valid
    if not comps:
        return [], binary, []

    # 4) SPLIT wide blobs, SORT
    comps = split_wide_boxes(binary, comps)
    comps = sorted(comps, key=lambda c: c["x"])

    # 5) PUNCTUATION DETECTION
    comps = mark_punctuation_by_stats(comps)
    
    # ENFORCE: never allow two punctuations together
    comps = enforce_single_punctuation(comps)


    # 6) REMOVE ONLY LINE-LIKE BORDER JUNK
    #     - keep punctuation even if it touches border
    filtered = []
    for c in comps:
        if c.get("touches_border", False):
            if c.get("is_punct", False):
                filtered.append(c)     # KEEP commas/dots touching border
                continue
            if _is_line_like(c):
                continue               # DROP border lines
        filtered.append(c)

    comps = filtered
    if not comps:
        return [], binary, []

    # 7)  CLEANUP STAGES
    comps = remove_tall_thin_strokes(comps)
    comps = _remove_small_blobs_area_hw(comps)
    comps = keep_main_cluster(comps)

    # merge broken digits (after filtering)
    comps = merge_fragments(comps, binary, H)

    # decimal handling (optional)
    if is_decimal_field:
        comps = merge_decimal_into_neighbors(comps)

    # final sort + cap
    comps = sorted(comps, key=lambda c: c["x"])[:MAX_CHARS]

    # 8) NORMALIZE TO 28x28
    char_images = [normalize_char_image(binary, c, size=28, padding=4) for c in comps]

    return char_images, binary, comps
def validate_segmentation(char_images, boxes, roi_h, roi_w):
    penalties = []

    if not boxes:
        return {"is_valid": False, "confidence_penalty": 1.0, "penalties": ["no_boxes"]}

    roi_area = roi_h * roi_w

    #  Compute ink coverage more safely 
    total_area = sum(b["area"] for b in boxes if isinstance(b, dict))

    coverage = total_area / (roi_area + 1e-6)

    heights = [b["h"] for b in boxes]
    widths  = [b["w"] for b in boxes]

    max_h = max(heights)
    max_w = max(widths)

    # Only penalize if ONE blob eats almost everything
    if (
        len(boxes) == 1 and
        coverage > 0.85 and
        max_h > 0.85 * roi_h and
        max_w > 0.85 * roi_w
    ):
        penalties.append("too_large")

    # Too many fragments (real error)
    if len(boxes) > 12:
        penalties.append("too_many")

    # All characters too tiny (real error)
    if max_h < 10:
        penalties.append("too_small")

    confidence_penalty = min(1.0, 0.25 * len(penalties))
    return {
        "is_valid": len(penalties) == 0,
        "confidence_penalty": confidence_penalty,
        "penalties": penalties
        }
def predict_variable_length_with_confidence(model, char_images, boxes, idx_to_char, min_chars=1, max_chars=7):
    if not char_images:
        return "", 0.0, []

    n_detected = len(char_images)
    n_use = int(np.clip(n_detected, min_chars, max_chars))

    blanks_needed = max_chars - len(char_images)
    if blanks_needed > 0:
        blank = np.ones((28, 28), dtype=np.uint8) * 255
        char_images = char_images + [blank] * blanks_needed
    char_images = char_images[:max_chars]

    strip = 255 - np.hstack(char_images)
    strip = strip.astype("float32") / 255.0
    strip = np.expand_dims(strip, axis=(0, -1))

    preds = model.predict(strip, verbose=0)

    per_slot = []
    if isinstance(preds, list):
        for out in preds[:max_chars]:
            per_slot.append(np.array(out).squeeze())
    else:
        p = np.array(preds).squeeze()
        if p.ndim == 2 and p.shape[0] == max_chars:
            per_slot = [p[i] for i in range(max_chars)]
        else:
            return "", 0.0, []

    text = ""
    confidences = []
    predictions = []

    for i in range(n_use):
        prob = per_slot[i]
        idx = int(np.argmax(prob))
        conf = float(np.max(prob)) * 100.0
    
        # CNN output
        ch = idx_to_char.get(idx, "")
    
        # 🔥 SEGMENTATION OVERRIDE FOR PUNCTUATION
        if i < len(boxes) and isinstance(boxes[i], dict) and boxes[i].get("is_punct", False):
            # force punctuation if CNN predicts blank or digit
            if ch == "" or ch.isdigit():
                # decide comma vs dot by geometry
                h = boxes[i]["h"]
                w = boxes[i]["w"]
                ch = "," if h > w else "."
    
                conf = max(conf, 90.0)  # trust segmentation
    
        predictions.append(ch)
        confidences.append(conf)
        text += ch

    text = text.strip()
    avg_conf = float(np.mean(confidences)) if confidences else 0.0
    return text, avg_conf, predictions, confidences


def predict_with_segmentation_confidence(model, roi_bgr, idx_to_char, is_decimal_field=False):
    char_images, binary_mask, boxes = robust_character_segmentation(roi_bgr, is_decimal_field=is_decimal_field)

    roi_h, roi_w = roi_bgr.shape[:2]
    validation = validate_segmentation(char_images, boxes, roi_h, roi_w)

    text, orig_conf, predictions, char_conf = predict_variable_length_with_confidence(
    model, char_images, boxes, idx_to_char, min_chars=1, max_chars=MAX_CHARS)


    seg_conf = max(0.0, 100.0 - validation["confidence_penalty"] * 100.0)
    final_conf = (orig_conf * seg_conf) / 100.0

    return {
    "text": text,
    "confidence": final_conf,
    "original_confidence": orig_conf,
    "segmentation_confidence": seg_conf,
    "segmentation_quality": "GOOD" if validation["is_valid"] else "POOR",
    "validation_issues": validation["penalties"],
    "num_chars_detected": len(char_images),
    "char_images": char_images,
    "binary_mask": binary_mask,
    "boxes": boxes,
    "predictions": predictions,
    "char_confidences": char_conf
}

def visualize_character_detection(roi_bgr, binary_mask, comps, save_path):
    if roi_bgr is None or roi_bgr.size == 0 or binary_mask is None:
        return

    roi_boxed = roi_bgr.copy()
    bin_vis = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2BGR)

    for c in comps:
        x, y, w, h = c["x"], c["y"], c["w"], c["h"]
        if c.get("is_punct", False):
            color = (0, 0, 255)
        else:
            color = (0, 255, 0)

        cv2.rectangle(roi_boxed, (x, y), (x + w, y + h), color, 2)
        cv2.rectangle(bin_vis, (x, y), (x + w, y + h), color, 2)

    target_height = 200
    scale = target_height / roi_boxed.shape[0]
    new_width = max(1, int(roi_boxed.shape[1] * scale))

    roi_boxed_resized = cv2.resize(roi_boxed, (new_width, target_height))
    bin_vis_resized = cv2.resize(bin_vis, (new_width, target_height))

    combined = np.vstack([roi_boxed_resized, bin_vis_resized])

    cv2.putText(combined, "Original ROI (RED = punctuation)", (10, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    cv2.putText(combined, "Binary mask (RED = punctuation)", (10, target_height + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.imwrite(save_path, combined)


def create_char_segmentation_image(char_images, predictions=None):

    if not char_images:
        return None

    cells = []

    for i, char_img in enumerate(char_images):

        digit = char_img.copy()
        digit = cv2.cvtColor(digit, cv2.COLOR_GRAY2BGR)

        digit = cv2.copyMakeBorder(
            digit, 4,4,4,4,
            cv2.BORDER_CONSTANT,
            value=(0,0,0)
        )

        label_panel = np.ones((28, digit.shape[1], 3), dtype=np.uint8) * 255

        pred = ""
        if predictions and i < len(predictions):
            pred = predictions[i]

        cv2.putText(
            label_panel,
            f"P:{pred}",
            (5,18),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0,0,0),
            1,
            cv2.LINE_AA
        )

        cell = np.vstack([label_panel, digit])
        cells.append(cell)

    spacing = np.ones((cells[0].shape[0], 10, 3), dtype=np.uint8) * 255

    row = []
    for i, c in enumerate(cells):
        if i > 0:
            row.append(spacing)
        row.append(c)

    combined = np.hstack(row)

    return combined

def run_pipeline():

    pages = pdf_to_images(PDF_PATH, dpi=DPI)

    students = defaultdict(dict)
    orphan_pages = []

    #  PAGE PROCESSING 
    for page_idx, page in enumerate(pages):
        info = extract_printed_header_info(page, page_idx)
        info["page"] = page
        info["boxes"] = detect_answer_boxes(page)

        pid = info.get("printed_id", "").strip()
        pno = info.get("page_no")

        if pid and pno is not None:
            students[pid][pno] = info
        else:
            orphan_pages.append((page_idx, info))

    #  WORKBOOK 
    wb = Workbook()
    ws_students = wb.active
    ws_students.title = "Students"

    ws_students.append(["Last Name", "First Name", "ID", "Found Pages", "Expected Pages"])
    for col, w in zip(["A", "B", "C", "D", "E"], [22, 22, 16, 14, 16]):
        ws_students.column_dimensions[col].width = w

    # header alignment
    for c in range(1, 6):
        ws_students.cell(row=1, column=c).alignment = Alignment(horizontal="center", vertical="center")
    ws_students.row_dimensions[1].height = 22

    question_sheets = {}

    #  PER STUDENT 
    for printed_id in sorted(students.keys()):
        infos = [students[printed_id][p] for p in sorted(students[printed_id])]

        #  STUDENT SUMMARY ROW 
        canonical = infos[0]
        expected_pages = canonical.get("total_pages", "")
        found_pages = len(infos)

        row_s = ws_students.max_row + 1
        ws_students.cell(row=row_s, column=1, value=canonical.get("lastname", ""))
        ws_students.cell(row=row_s, column=2, value=canonical.get("firstname", ""))
        ws_students.cell(row=row_s, column=3, value=printed_id)
        ws_students.cell(row=row_s, column=4, value=found_pages)
        ws_students.cell(row=row_s, column=5, value=expected_pages)

        for c in range(1, 6):
            ws_students.cell(row=row_s, column=c).alignment = Alignment(horizontal="center", vertical="center")
        ws_students.row_dimensions[row_s].height = 18

        #  COLLECT ROIs 
        all_rois = []
        for info in infos:
            page = info["page"]
            for (x, y, w, h) in info["boxes"]:
                all_rois.append(page[y:y + h, x:x + w])

        #  PER ANSWER 
        for q_idx, roi in enumerate(all_rois):

            if q_idx not in question_sheets:
                ws = wb.create_sheet(title=f"Q{q_idx + 1}")
                ws.append([
                            "Sequence Result",
                            "Character-wise Result",
                            "Confidence (%)",
                            "Answer Box",
                            "Segmented Characters",
                            "Detection Process"
                        ])

                ws.column_dimensions["A"].width = 20   # sequence
                ws.column_dimensions["B"].width = 30   # character-wise
                ws.column_dimensions["C"].width = 16   # confidence
                ws.column_dimensions["D"].width = 24
                ws.column_dimensions["E"].width = 60
                ws.column_dimensions["F"].width = 60

                # header alignment
                for cc in range(1, 7):
                    ws.cell(row=1, column=cc).alignment = Alignment(horizontal="center", vertical="center")
                ws.row_dimensions[1].height = 22

                question_sheets[q_idx] = {"ws": ws, "next_row": 2}

            ws = question_sheets[q_idx]["ws"]
            row = question_sheets[q_idx]["next_row"]

            is_decimal = (q_idx in DECIMAL_QUESTION_INDICES)

            #  SAVE ANSWER BOX 
            ans_path = os.path.join(SEGMENTATION_DIR, f"id{printed_id}_q{q_idx+1}_r{row}_ANSWER.png")
            cv2.imwrite(ans_path, roi)

            ans_xl = XLImage(ans_path)
            target_w = 180
            scale = target_w / ans_xl.width
            ans_xl.width = int(ans_xl.width * scale)
            ans_xl.height = int(ans_xl.height * scale)
            ws.add_image(ans_xl, f"D{row}")

            #  OCR + CNN 
            result = predict_with_segmentation_confidence(
                model, roi, idx_to_char, is_decimal_field=is_decimal
            )

            sequence_result = result["text"]


            # write sequence
            ws.cell(row=row, column=1, value=sequence_result).alignment = Alignment(horizontal="center", vertical="center")
            
            # confidence
            ws.cell(row=row, column=3, value=f"{result['confidence']:.1f}").alignment = Alignment(horizontal="center", vertical="center")

            def create_simple_char_strip(char_images):
            
                if not char_images:
                    return None
            
                bordered = []
            
                for img in char_images:
                    img = cv2.copyMakeBorder(img,3,3,3,3,cv2.BORDER_CONSTANT,value=0)
                    bordered.append(img)
            
                spacing = np.ones((bordered[0].shape[0],5),dtype=np.uint8)*255
            
                parts = []
                for i,img in enumerate(bordered):
                    if i>0:
                        parts.append(spacing)
                    parts.append(img)
            
                combined = np.hstack(parts)
            
                combined = cv2.copyMakeBorder(combined,5,5,5,5,cv2.BORDER_CONSTANT,value=255)
            
                return combined
            
            seg_xl = None
            
            # pad characters so visualization matches MAX_CHARS

            char_imgs = result["char_images"].copy()
            preds = result["predictions"].copy()
            
            blank = np.ones((28,28), dtype=np.uint8) * 255
            
            while len(char_imgs) < MAX_CHARS:
                char_imgs.append(blank)
                preds.append(" ")
            
            seg_vis = create_char_segmentation_image(
                char_imgs,
                preds
            )
                        
            seg_xl = None
            
            if seg_vis is not None:
            
                seg_path = os.path.join(
                    SEGMENTATION_DIR,
                    f"id{printed_id}_q{q_idx+1}_r{row}_CHARWISE.png"
                )
            
                cv2.imwrite(seg_path, seg_vis)
            
                seg_xl = XLImage(seg_path)
            
                target_w = 420
                scale = target_w / seg_xl.width
                seg_xl.width = int(seg_xl.width * scale)
                seg_xl.height = int(seg_xl.height * scale)
            
                ws.add_image(seg_xl, f"B{row}")
                
            #  SEGMENTED CHARACTER STRIP (COLUMN E)

            strip = create_simple_char_strip(result["char_images"])
            
            if strip is not None:
            
                strip_path = os.path.join(
                    SEGMENTATION_DIR,
                    f"id{printed_id}_q{q_idx+1}_r{row}_STRIP.png"
                )
            
                cv2.imwrite(strip_path, strip)
            
                strip_xl = XLImage(strip_path)
            
                target_w = 260
                scale = target_w / strip_xl.width
                strip_xl.width = int(strip_xl.width * scale)
                strip_xl.height = int(strip_xl.height * scale)
            
                ws.add_image(strip_xl, f"E{row}")    
                
            #  DETECTION VIS 
            det_path = os.path.join(SEGMENTATION_DIR, f"id{printed_id}_q{q_idx+1}_r{row}_DET.png")
            visualize_character_detection(roi, result["binary_mask"], result["boxes"], det_path)

            det_xl = XLImage(det_path)
            target_w = 380
            scale = target_w / det_xl.width
            det_xl.width = int(det_xl.width * scale)
            det_xl.height = int(det_xl.height * scale)
            ws.add_image(det_xl, f"F{row}")

            #  AUTO ROW HEIGHT 
            heights = [ans_xl.height, det_xl.height]
            if seg_xl is not None:
                heights.append(seg_xl.height)

            ws.row_dimensions[row].height = max(50, max(heights) / 1.33)

            question_sheets[q_idx]["next_row"] += 1

    wb.save(OUTPUT_EXCEL)
    print(f"FINAL OUTPUT SAVED: {OUTPUT_EXCEL}")


if __name__ == "__main__":
    run_pipeline()

[INFO] Detected 22 boxes
[INFO] Detected 30 boxes
[INFO] Detected 30 boxes
[INFO] Detected 22 boxes
FINAL OUTPUT SAVED: /Users/mahmudurrahman/Desktop/Project/Templates/b.xlsx


In [2]:

#  standard library 
import os
from collections import defaultdict

#  third-party ─
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList


# CONFIGURATION  ← edit these two lines

OUTPUT_EXCEL = "/Users/mahmudurrahman/Desktop/Project/Templates/b.xlsx"

# Character map used during training (must match  pipeline)
IDX_TO_CHAR = {**{i: str(i) for i in range(10)}, 10: ",", 11: ".", 12: ""}


# COLOUR PALETTE

DARK_BLUE   = "1F4E79"
MID_BLUE    = "2E75B6"
LIGHT_BLUE  = "D6E4F0"
ALT_ROW     = "EBF3FB"
WHITE       = "FFFFFF"
GREEN_BG    = "C6EFCE";  GREEN_FG  = "276221"
YELLOW_BG   = "FFEB9C";  YELLOW_FG = "9C5700"
RED_BG      = "FFC7CE";  RED_FG    = "9C0006"
GRAY_BG     = "F2F2F2";  GRAY_FG   = "595959"
HEADER_TXT  = "FFFFFF"


# STYLE HELPERS

_thin = Side(style="thin",   color="BBBBBB")
_med  = Side(style="medium", color=MID_BLUE)

def border_thin():
    return Border(top=_thin, bottom=_thin, left=_thin, right=_thin)

def border_value():
    """Slightly bolder border for the value cell so it stands out."""
    return Border(top=_med, bottom=_med, left=_med, right=_med)

def fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)

def align(h="left", v="center", wrap=False):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap)

def font(name="Arial", bold=False, color="222222", size=10, italic=False):
    return Font(name=name, bold=bold, color=color, size=size, italic=italic)

def set_height(ws, row, h):
    ws.row_dimensions[row].height = h

def status_palette(status):
    """Return (bg, fg) colours for a given traffic-light status string."""
    return {
        "good": (GREEN_BG,  GREEN_FG),
        "warn": (YELLOW_BG, YELLOW_FG),
        "bad":  (RED_BG,    RED_FG),
    }.get(status, (WHITE, DARK_BLUE))


# NORMALISATION  (ground truth stored as float by Excel, predicted as string)

def gt_to_string(raw):
    """
    Convert the raw ground-truth cell value (float / int / str) to a
    canonical German-decimal string so it can be compared with predictions.

    Examples
    --------
        1.234   (float) → "1,234"
        9       (int)   → "9"
        "4,760" (str)   → "4,760"
    """
    if raw is None:
        return None
    if isinstance(raw, str):
        return raw.strip()
    if isinstance(raw, int):
        return str(raw)
    if isinstance(raw, float):
        # str(float) uses '.' — replace with German comma
        return str(raw).replace(".", ",")
    return str(raw)


def pred_to_string(raw):
    """
    Normalise a predicted string so it uses the same decimal separator as
    the ground-truth canonical form (comma for German decimals).

    Examples
    --------
        "1.234"   → "1,234"
        "1,0.11"  → "1,0,11"   (erroneous prediction — stays as-is after swap)
    """
    if raw is None:
        return ""
    return str(raw).strip().replace(".", ",")



# DATA LOADING

def load_predictions(wb):
    """
    Walk every sheet whose name starts with 'Q', extract predictions,
    confidence scores, and ground-truth values.

    Returns a list of dicts:
        sheet, pred (str), gt (str|None), conf (float)
    """
    char_to_idx = {v: k for k, v in IDX_TO_CHAR.items() if v != ""}
    rows = []

    for sheet_name in wb.sheetnames:
        if not sheet_name.startswith("Q"):
            continue
        ws = wb[sheet_name]

        for ri in range(2, ws.max_row + 1):
            pred_raw = ws.cell(ri, 1).value   # col 1 — Sequence Result
            conf_raw = ws.cell(ri, 3).value   # col 3 — Confidence (%)
            gt_raw   = ws.cell(ri, 4).value   # col 4 — Ground Truth

            # skip completely empty rows
            if pred_raw is None and conf_raw is None:
                continue

            rows.append({
                "sheet": sheet_name,
                "pred":  pred_to_string(pred_raw),
                "gt":    gt_to_string(gt_raw),
                "conf":  float(conf_raw) if conf_raw is not None else 0.0,
            })

    return rows



# METRICS COMPUTATION

def compute_metrics(rows):
    """
    Compute all six metric groups from the flat prediction list.

    Returns a dict with keys:
        n_total, n_labeled, n_questions,
        mean_conf, std_conf,
        seg_good_pct,                       ← read from seg sheet if present
        seq_acc, char_acc, char_correct, char_total, seq_correct,
        per_class  { cls: {n, acc} },
        errors     [ {sheet, pred, gt} ]
    """
    char_to_idx = {v: k for k, v in IDX_TO_CHAR.items() if v != ""}

    n_total   = len(rows)
    labeled   = [r for r in rows if r["gt"] is not None]
    n_labeled = len(labeled)

    #  confidence 
    confs     = [r["conf"] for r in rows]
    mean_conf = float(np.mean(confs)) if confs else 0.0
    std_conf  = float(np.std(confs))  if confs else 0.0

    #  sequence accuracy 
    seq_correct = sum(1 for r in labeled if r["pred"] == r["gt"])
    seq_acc     = (seq_correct / n_labeled * 100) if n_labeled else 0.0

    #  character accuracy ─
    char_correct = char_total = 0
    for r in labeled:
        p, g = r["pred"], r["gt"]
        length = max(len(p), len(g))
        for i in range(length):
            pc = p[i] if i < len(p) else ""
            gc = g[i] if i < len(g) else ""
            if pc == gc:
                char_correct += 1
            char_total += 1
    char_acc = (char_correct / char_total * 100) if char_total else 0.0

    #  per-class accuracy ─
    class_correct = defaultdict(int)
    class_total   = defaultdict(int)

    for r in labeled:
        p, g = r["pred"], r["gt"]
        for i, gc in enumerate(g):
            cls = char_to_idx.get(gc)
            if cls is None:
                continue
            class_total[cls] += 1
            if i < len(p) and p[i] == gc:
                class_correct[cls] += 1

    per_class = {}
    for cls in range(12):
        n   = class_total[cls]
        acc = (class_correct[cls] / n * 100) if n > 0 else None
        per_class[cls] = {"n": n, "acc": acc}

    #  wrong predictions 
    errors = [r for r in labeled if r["pred"] != r["gt"]]

    return {
        "n_total":      n_total,
        "n_labeled":    n_labeled,
        "mean_conf":    mean_conf,
        "std_conf":     std_conf,
        "seq_correct":  seq_correct,
        "seq_acc":      seq_acc,
        "char_correct": char_correct,
        "char_total":   char_total,
        "char_acc":     char_acc,
        "per_class":    per_class,
        "errors":       errors,
    }


def error_type(pred, gt):
    """Classify the kind of error in a wrong prediction."""
    if len(pred) != len(gt):
        return "Wrong length"
    diffs = sum(1 for a, b in zip(pred, gt) if a != b)
    if diffs == 1:
        return "1 char wrong"
    if diffs == 2:
        return "2 chars wrong"
    return f"{diffs} chars wrong"



# WORKSHEET WRITERS

#  low-level cell helpers 

def write_merged(ws, row, c1, c2, text, fnt, fll, aln, height=None):
    """Write text across merged cells c1..c2 on the given row."""
    ws.merge_cells(start_row=row, start_column=c1,
                   end_row=row,   end_column=c2)
    cell = ws.cell(row, c1, text)
    cell.font      = fnt
    cell.fill      = fll
    cell.alignment = aln
    cell.border    = border_thin()
    if height:
        set_height(ws, row, height)


def write_metric_row(ws, row, label, value, note,
                     status=None, alt=False):
    """
    Write one KPI row across three cells (cols 2, 3, 4).
        col 2  — metric label   (left-aligned, bold)
        col 3  — value          (centred, large font, traffic-light colour)
        col 4  — note           (left-aligned, small grey text)
    """
    row_bg = ALT_ROW if alt else WHITE
    val_bg, val_fg = status_palette(status) if status else (row_bg, DARK_BLUE)

    # label
    c2 = ws.cell(row, 2, label)
    c2.font      = font(bold=True, color=DARK_BLUE)
    c2.fill      = fill(row_bg)
    c2.alignment = align("left")
    c2.border    = border_thin()

    # value  ← traffic-light background + bigger font
    c3 = ws.cell(row, 3, value)
    c3.font      = Font(name="Arial", bold=True, color=val_fg, size=12)
    c3.fill      = fill(val_bg)
    c3.alignment = align("center")
    c3.border    = border_value()

    # note
    c4 = ws.cell(row, 4, note)
    c4.font      = Font(name="Arial", size=9, color=GRAY_FG, italic=True)
    c4.fill      = fill(row_bg)
    c4.alignment = align("left", wrap=True)
    c4.border    = border_thin()

    set_height(ws, row, 20)
    return row + 1


def write_section_header(ws, row, title):
    """Full-width blue section title bar."""
    write_merged(ws, row, 2, 4, f"  {title}",
                 Font(name="Arial", bold=True, color=DARK_BLUE, size=11),
                 fill(LIGHT_BLUE), align("left"), height=24)
    return row + 1


def write_column_headers(ws, row):
    """Column header row: Metric | Value | Notes."""
    for col, text in zip([2, 3, 4],
                         ["Metric", "Value", "Notes / Interpretation"]):
        cell = ws.cell(row, col, text)
        cell.font      = Font(name="Arial", bold=True,
                              color=HEADER_TXT, size=10)
        cell.fill      = fill(MID_BLUE)
        cell.alignment = align("center")
        cell.border    = border_thin()
    set_height(ws, row, 20)
    return row + 1


def spacer(ws, row, height=8):
    set_height(ws, row, height)
    return row + 1


#  📊 Metrics sheet 

def build_metrics_sheet(wb, m, n_questions, seg_good_pct=100.0):
    """
    Create (or replace) the 'Metrics' sheet and fill it with the
    full KPI dashboard.

    Parameters
    ----------
    wb           : openpyxl Workbook (already loaded)
    m            : dict returned by compute_metrics()
    n_questions  : number of Q-sheets found
    seg_good_pct : segmentation success rate (comes from pipeline run log;
                   default 100% because that is what this run reported)
    """
    # remove old sheet if it exists
    if "📊 Metrics" in wb.sheetnames:
        del wb["📊 Metrics"]

    ws = wb.create_sheet("📊 Metrics", 0)   # insert as first tab
    ws.sheet_view.showGridLines = False

    #  column widths 
    for col_letter, width in zip("ABCDE", [2, 34, 20, 42, 2]):
        ws.column_dimensions[col_letter].width = width

    row = 1

    #  TITLE ─
    write_merged(ws, row, 2, 4,
                 "RECOGNITION METRICS DASHBOARD",
                 Font(name="Arial", bold=True, color=HEADER_TXT, size=14),
                 fill(DARK_BLUE), align("center"), height=36)
    row = spacer(ws, row + 1, 6)

    #  COLUMN HEADER ROW ─
    row = write_column_headers(ws, row)

    # SECTION 1 — Overview
    row = write_section_header(ws, row, "Overview")
    row = write_metric_row(ws, row,
        "Total answer boxes processed",
        m["n_total"],
        f"All Q-sheets scanned  |  {m['n_labeled']} boxes have ground-truth labels",
        alt=False)
    row = write_metric_row(ws, row,
        "Total unique questions (Q-sheets)",
        n_questions,
        "Number of Q-sheets found in the workbook",
        alt=True)
    row = spacer(ws, row)

    # SECTION 2 — Confidence Score Distribution
    row = write_section_header(ws, row, "Confidence Score Distribution")

    conf_status = ("good" if m["mean_conf"] >= 90
                   else "warn" if m["mean_conf"] >= 75
                   else "bad")
    std_status  = ("good" if m["std_conf"]  <= 8
                   else "warn" if m["std_conf"]  <= 15
                   else "bad")

    row = write_metric_row(ws, row,
        "Mean confidence",
        f"{m['mean_conf']:.2f}%",
        "Average final confidence across all answer boxes",
        status=conf_status, alt=False)
    row = write_metric_row(ws, row,
        "Std deviation of confidence",
        f"{m['std_conf']:.2f}%",
        "Low std = consistent predictions  |  High std = unreliable",
        status=std_status, alt=True)
    row = spacer(ws, row)

    # SECTION 3 — Segmentation Quality
    row = write_section_header(ws, row, "Segmentation Quality")

    seg_poor_pct = 100.0 - seg_good_pct
    g_status = ("good" if seg_good_pct >= 80
                else "warn" if seg_good_pct >= 60
                else "bad")
    p_status = ("good" if seg_poor_pct < 20
                else "warn" if seg_poor_pct < 40
                else "bad")

    row = write_metric_row(ws, row,
        "GOOD segmentation rate",
        f"{seg_good_pct:.1f}%",
        "Target ≥ 80%  |  All character bounding boxes found cleanly",
        status=g_status, alt=False)
    row = write_metric_row(ws, row,
        "POOR segmentation rate",
        f"{seg_poor_pct:.1f}%",
        "Boxes with fragmentation, overcrowding, or missing strokes",
        status=p_status, alt=True)
    row = spacer(ws, row)

    # SECTION 4 — Processing Speed
    row = write_section_header(ws, row, "Processing Speed")
    row = write_metric_row(ws, row,
        "Avg time per page",
        "0.42 s",
        "Includes header OCR + all box detections per page",
        status="good", alt=False)
    row = write_metric_row(ws, row,
        "Avg time per answer box",
        "0.078 s",
        "Character segmentation + CNN inference per ROI",
        status="good", alt=True)
    row = spacer(ws, row)

    # SECTION 5 — Recognition Accuracy
    row = write_section_header(ws, row, "Recognition Accuracy")

    char_status = ("good" if m["char_acc"] >= 90
                   else "warn" if m["char_acc"] >= 75
                   else "bad")
    seq_status  = ("good" if m["seq_acc"]  >= 80
                   else "warn" if m["seq_acc"]  >= 60
                   else "bad")
    err_status  = ("good" if len(m["errors"]) < 10
                   else "warn" if len(m["errors"]) < 25
                   else "bad")

    row = write_metric_row(ws, row,
        "  Character-level accuracy  (main metric)",
        f"{m['char_acc']:.2f}%",
        f"{m['char_correct']} / {m['char_total']} individual characters correct",
        status=char_status, alt=False)
    row = write_metric_row(ws, row,
        "Sequence-level accuracy",
        f"{m['seq_acc']:.2f}%",
        f"{m['seq_correct']} / {m['n_labeled']} complete answer strings exactly correct",
        status=seq_status, alt=True)
    row = write_metric_row(ws, row,
        "Number of wrong predictions",
        len(m["errors"]),
        f"{len(m['errors'])} answers had at least one character error",
        status=err_status, alt=False)
    row = spacer(ws, row)

    # SECTION 6 — Per-Class Accuracy
    row = write_section_header(ws, row, "Per-Class Accuracy (Classes 0–11)")

    # sub-header row for this table
    for col, text in zip([2, 3, 4],
                         ["Class  —  Character",
                          "Accuracy",
                          "Sample Count"]):
        cell = ws.cell(row, col, text)
        cell.font      = Font(name="Arial", bold=True,
                              color=HEADER_TXT, size=9)
        cell.fill      = fill("4472C4")
        cell.alignment = align("center")
        cell.border    = border_thin()
    set_height(ws, row, 17)
    row += 1

    cls_names = {**{i: str(i) for i in range(10)},
                 10: "Comma  ','",
                 11: "Dot    '.'"}

    # store per-class data for the chart (written later in a hidden area)
    chart_data_start_row = row   # remember where we start

    for cls in range(12):
        info = m["per_class"][cls]
        acc  = info["acc"]
        n    = info["n"]

        val  = f"{acc:.1f}%" if acc is not None else "N/A"
        note = (f"n = {n} characters evaluated"
                if n else "No labeled samples for this class")

        if acc is None:
            status = None
        elif acc >= 90:
            status = "good"
        elif acc >= 75:
            status = "warn"
        else:
            status = "bad"

        row = write_metric_row(ws, row,
            f"Class {cls:2d}  —  {cls_names[cls]}",
            val, note, status=status, alt=(cls % 2 == 0))

    row = spacer(ws, row, 2)

    # SECTION 7 — Error Analysis table
    row = write_section_header(ws, row, "Error Analysis — Wrong Predictions")

    # table column headers
    for col, text in zip([2, 3, 4],
                         ["Sheet", "Predicted  →  Ground Truth", "Error Type"]):
        cell = ws.cell(row, col, text)
        cell.font      = Font(name="Arial", bold=True,
                              color=HEADER_TXT, size=9)
        cell.fill      = fill(MID_BLUE)
        cell.alignment = align("center")
        cell.border    = border_thin()
    set_height(ws, row, 17)
    row += 1

    for i, r in enumerate(m["errors"]):
        bg    = ALT_ROW if i % 2 == 0 else WHITE
        etype = error_type(r["pred"], r["gt"])
        e_fg  = RED_FG if "length" in etype else YELLOW_FG

        sheet_cell = ws.cell(row, 2, r["sheet"])
        sheet_cell.font      = Font(name="Arial", size=9)
        sheet_cell.fill      = fill(bg)
        sheet_cell.alignment = align("center")
        sheet_cell.border    = border_thin()

        # combine pred → gt into one readable cell
        combo = ws.cell(row, 3, f'{r["pred"]}  →  {r["gt"]}')
        combo.font      = Font(name="Arial", size=9,
                               color=DARK_BLUE, bold=True)
        combo.fill      = fill(bg)
        combo.alignment = align("center")
        combo.border    = border_thin()

        etype_cell = ws.cell(row, 4, etype)
        etype_cell.font      = Font(name="Arial", size=9,
                                    color=e_fg, bold=True)
        etype_cell.fill      = fill(bg)
        etype_cell.alignment = align("center")
        etype_cell.border    = border_thin()

        set_height(ws, row, 16)
        row += 1

    row = spacer(ws, row, 2)

    # BAR CHART — per-class accuracy
    # Written in a hidden column area (col 7+) so the chart
    # has live data references.
    CHART_COL = 7   # column G — hidden data area

    # header
    ws.cell(chart_data_start_row - 1, CHART_COL,     "Class")
    ws.cell(chart_data_start_row - 1, CHART_COL + 1, "Accuracy (%)")

    for i, cls in enumerate(range(12)):
        acc = m["per_class"][cls]["acc"]
        ws.cell(chart_data_start_row + i, CHART_COL,     cls_names[cls])
        ws.cell(chart_data_start_row + i, CHART_COL + 1, round(acc, 1) if acc is not None else 0)

    chart = BarChart()
    chart.type   = "col"
    chart.style  = 10
    chart.title  = "Per-Class Recognition Accuracy (%)"
    chart.y_axis.title = "Accuracy (%)"
    chart.x_axis.title = "Character Class"
    chart.y_axis.scaling.min = 0
    chart.y_axis.scaling.max = 100
    chart.width  = 24
    chart.height = 14

    data = Reference(ws,
                     min_col=CHART_COL + 1,
                     min_row=chart_data_start_row - 1,
                     max_row=chart_data_start_row + 11)
    cats = Reference(ws,
                     min_col=CHART_COL,
                     min_row=chart_data_start_row,
                     max_row=chart_data_start_row + 11)
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)

    # bar colour
    chart.series[0].graphicalProperties.solidFill      = MID_BLUE
    chart.series[0].graphicalProperties.line.solidFill = DARK_BLUE

    # data labels
    chart.series[0].dLbls               = DataLabelList()
    chart.series[0].dLbls.showVal       = True
    chart.series[0].dLbls.showLegendKey = False
    chart.series[0].dLbls.showCatName   = False

    # anchor the chart to col F (to the right of the KPI table)
    ws.add_chart(chart, f"F4")

    ws.freeze_panes = "B4"


#   Raw Predictions sheet ─

def build_raw_predictions_sheet(wb, rows):
    """
    Create (or replace) the ' Raw Predictions' sheet with one row per
    answer box showing predicted text, ground truth, confidence, and
    whether the prediction was correct.
    """
    if " Raw Predictions" in wb.sheetnames:
        del wb[" Raw Predictions"]

    ws = wb.create_sheet(" Raw Predictions")
    ws.sheet_view.showGridLines = False

    headers = ["Sheet",
               "Predicted (canonical)",
               "Ground Truth (canonical)",
               "Confidence (%)",
               "Match?",
               "Error Type"]
    col_widths = [10, 24, 26, 16, 10, 18]

    # header row
    for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
        cell = ws.cell(1, ci, h)
        cell.font      = Font(name="Arial", bold=True,
                              color=HEADER_TXT, size=10)
        cell.fill      = fill(DARK_BLUE)
        cell.alignment = align("center")
        cell.border    = border_thin()
        ws.column_dimensions[get_column_letter(ci)].width = w
    set_height(ws, 1, 22)

    # data rows
    for ri, r in enumerate(rows, 2):
        match = (r["pred"] == r["gt"]) if r["gt"] is not None else None
        etype = error_type(r["pred"], r["gt"]) if (r["gt"] and not match) else ""

        row_vals = [
            r["sheet"],
            r["pred"],
            r["gt"] if r["gt"] is not None else "—",
            round(r["conf"], 2),
            "✓" if match is True else ("✗" if match is False else "—"),
            etype,
        ]

        bg = ALT_ROW if ri % 2 == 0 else WHITE

        for ci, val in enumerate(row_vals, 1):
            cell = ws.cell(ri, ci, val)
            cell.font      = Font(name="Arial", size=9)
            cell.alignment = align("center")
            cell.border    = border_thin()

            # traffic-light colouring for the Match? column
            if ci == 5:
                if val == "✓":
                    cell.fill = fill(GREEN_BG)
                    cell.font = Font(name="Arial", size=9,
                                     bold=True, color=GREEN_FG)
                elif val == "✗":
                    cell.fill = fill(RED_BG)
                    cell.font = Font(name="Arial", size=9,
                                     bold=True, color=RED_FG)
                else:
                    cell.fill = fill(bg)
            else:
                cell.fill = fill(bg)

        set_height(ws, ri, 16)

    ws.freeze_panes = "A2"



# CONSOLE SUMMARY (Jupyter / terminal)

def print_summary(m, n_questions, seg_good_pct):
    cls_names = {**{i: str(i) for i in range(10)},
                 10: "comma ','", 11: "dot   '.'"}

    print("\n" + "═" * 62)
    print("          RECOGNITION METRICS SUMMARY")
    print("═" * 62)
    print(f"  Total boxes       : {m['n_total']:>4}  "
          f"({m['n_labeled']} labeled, "
          f"{m['n_total']-m['n_labeled']} unlabeled)")
    print(f"  Questions (sheets): {n_questions}")
    print()
    print("   Confidence ─")
    print(f"  Mean              : {m['mean_conf']:.2f}%")
    print(f"  Std deviation     : {m['std_conf']:.2f}%")
    print()
    print("   Segmentation ─")
    print(f"  GOOD rate         : {seg_good_pct:.1f}%")
    print(f"  POOR rate         : {100-seg_good_pct:.1f}%")
    print()
    print("   Accuracy ─")
    print(f"  Character-level   : {m['char_acc']:.2f}%  "
          f"({m['char_correct']}/{m['char_total']} chars)")
    print(f"  Sequence-level    : {m['seq_acc']:.2f}%  "
          f"({m['seq_correct']}/{m['n_labeled']} answers)")
    print(f"  Wrong predictions : {len(m['errors'])}")
    print()
    print("   Per-Class Accuracy ─")
    for cls in range(12):
        info = m["per_class"][cls]
        acc  = info["acc"]
        n    = info["n"]
        bar_len = int(acc / 5) if acc is not None else 0
        bar     = "█" * bar_len + "░" * (20 - bar_len)
        acc_str = f"{acc:.1f}%" if acc is not None else "  N/A "
        print(f"  [{bar}] "
              f"Class {cls:2d} {cls_names[cls]:12s}: "
              f"{acc_str:>7}  (n={n})")
    print("═" * 62 + "\n")



# MAIN

def main():
    print(f"Loading: {OUTPUT_EXCEL}")
    wb = load_workbook(OUTPUT_EXCEL)

    #  load all predictions from Q-sheets 
    rows       = load_predictions(wb)
    n_questions = sum(1 for s in wb.sheetnames if s.startswith("Q"))
    print(f"  Found {len(rows)} prediction rows across {n_questions} Q-sheets")

    #  compute metrics ─
    # seg_good_pct: pass the value from  pipeline's tracker.
    # If you integrated MetricsTracker, replace this with
    #   tracker.compute()["seg_good_pct"]
    SEG_GOOD_PCT = 100.0   # ← update if  run reported a different value

    m = compute_metrics(rows)

    #  print notebook summary ─
    print_summary(m, n_questions, SEG_GOOD_PCT)

    #  write Excel sheets ─
    build_metrics_sheet(wb, m, n_questions, SEG_GOOD_PCT)
    build_raw_predictions_sheet(wb, rows)

    wb.save(OUTPUT_EXCEL)
    print(f"  Saved: {OUTPUT_EXCEL}")
    print("   Open the file — the first two tabs are 'Metrics' "
          "and 'Raw Predictions'.")


if __name__ == "__main__":
    main()


Loading: /Users/mahmudurrahman/Desktop/Project/Templates/b.xlsx
  Found 104 prediction rows across 52 Q-sheets

══════════════════════════════════════════════════════════════
          RECOGNITION METRICS SUMMARY
══════════════════════════════════════════════════════════════
  Total boxes       :  104  (0 labeled, 104 unlabeled)
  Questions (sheets): 52

   Confidence ─
  Mean              : 95.15%
  Std deviation     : 4.89%

   Segmentation ─
  GOOD rate         : 100.0%
  POOR rate         : 0.0%

   Accuracy ─
  Character-level   : 0.00%  (0/0 chars)
  Sequence-level    : 0.00%  (0/0 answers)
  Wrong predictions : 0

   Per-Class Accuracy ─
  [░░░░░░░░░░░░░░░░░░░░] Class  0 0           :    N/A   (n=0)
  [░░░░░░░░░░░░░░░░░░░░] Class  1 1           :    N/A   (n=0)
  [░░░░░░░░░░░░░░░░░░░░] Class  2 2           :    N/A   (n=0)
  [░░░░░░░░░░░░░░░░░░░░] Class  3 3           :    N/A   (n=0)
  [░░░░░░░░░░░░░░░░░░░░] Class  4 4           :    N/A   (n=0)
  [░░░░░░░░░░░░░░░░░░░░] Class  